# Online Retail II Exploration

This notebook loads both yearly sheets from the Online Retail II workbook into pandas, validates and prepares the data, flags questionable rows, and saves reviewable outputs.

## 1. Import Required Libraries

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 2. Configure File Paths

In [2]:
PROJECT_ROOT = Path.cwd()
INPUT_PATH = PROJECT_ROOT / "data" / "online_retail_II.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "reports"

INPUT_PATH, OUTPUT_DIR

(WindowsPath('c:/Users/sezen/Projects/RetailDemo/retail-self-refreshing-report/data/online_retail_II.xlsx'),
 WindowsPath('c:/Users/sezen/Projects/RetailDemo/retail-self-refreshing-report/reports'))

## 3. Read Excel Workbook into a DataFrame

In [3]:
sheets = pd.read_excel(INPUT_PATH, sheet_name=None)
df = pd.concat(
    [sheet.assign(source_sheet=sheet_name) for sheet_name, sheet in sheets.items()],
    ignore_index=True,
)

print(f"Loaded {len(sheets)} sheets and {len(df):,} rows.")
df.head()

Loaded 2 sheets and 1,067,371 rows.


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Year 2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Year 2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010


## 3b. Remove Duplicate Rows from Overlapping Sheet Date Ranges

In [4]:
# The "Year 2009-2010" sheet runs through 2010-12-09, overlapping the first 9 days of the
# "Year 2010-2011" sheet (which starts 2010-12-01). Invoices in that window are recorded
# identically in both sheets, so concatenating them as-is double-counts that period's revenue.
overlap_invoices = (
    df.groupby("Invoice")["source_sheet"].nunique().loc[lambda s: s > 1].index
)
overlap_row_mask = df["Invoice"].isin(overlap_invoices) & (df["source_sheet"] == "Year 2009-2010")

print(f"Invoices present in both sheets: {len(overlap_invoices):,}")
print(f"Rows dropped (the 'Year 2009-2010' copy of those invoices): {overlap_row_mask.sum():,}")

df = df.loc[~overlap_row_mask].reset_index(drop=True)
print(f"Rows remaining: {len(df):,}")

Invoices present in both sheets: 1,088
Rows dropped (the 'Year 2009-2010' copy of those invoices): 22,523
Rows remaining: 1,044,848


## 4. Inspect the DataFrame

In [5]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.info()
df.head(10)

Shape: (1044848, 9)
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'source_sheet']
<class 'pandas.DataFrame'>
RangeIndex: 1044848 entries, 0 to 1044847
Data columns (total 9 columns):
 #   Column        Non-Null Count    Dtype         
---  ------        --------------    -----         
 0   Invoice       1044848 non-null  object        
 1   StockCode     1044848 non-null  object        
 2   Description   1040573 non-null  object        
 3   Quantity      1044848 non-null  int64         
 4   InvoiceDate   1044848 non-null  datetime64[us]
 5   Price         1044848 non-null  float64       
 6   Customer ID   809561 non-null   float64       
 7   Country       1044848 non-null  str           
 8   source_sheet  1044848 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(2)
memory usage: 71.7+ MB


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Year 2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Year 2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom,Year 2009-2010
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom,Year 2009-2010
8,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom,Year 2009-2010
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,13085.0,United Kingdom,Year 2009-2010


## 5. Validate Expected Columns

In [6]:
expected_columns = {
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer ID",
    "Country",
    "source_sheet",
}
missing_columns = expected_columns.difference(df.columns)
extra_columns = set(df.columns).difference(expected_columns)

assert not missing_columns, f"Missing columns: {sorted(missing_columns)}"
print("Schema is valid.")
print("Extra columns:", sorted(extra_columns) if extra_columns else "None")

Schema is valid.
Extra columns: None


## 6. Convert Data Types

In [7]:
# Invoice and StockCode mix int and str values in the source file (purely-numeric values get
# read as ints, e.g. 492525 or 85123, while anything needing a letter -- "C489449", "85123A",
# "AMAZONFEE" -- comes in as str). Casting both to string once, here, means every downstream
# comparison/regex/prefix-check on these columns behaves consistently for the rest of the notebook.
df["Invoice"] = df["Invoice"].astype("string")
df["StockCode"] = df["StockCode"].astype("string")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Revenue"] = df["Quantity"] * df["Price"]

df[["Invoice", "StockCode", "Quantity", "Price", "InvoiceDate", "Revenue"]].dtypes

Invoice                string
StockCode              string
Quantity                int64
Price                 float64
InvoiceDate    datetime64[us]
Revenue               float64
dtype: object

## 6b. Remove Literal Test Rows (`TEST001` / `TEST002`)

In [ ]:
# TEST001/TEST002 are literal QA/test rows (Description "This is a test product."), not real
# transactions -- they show up scattered across normal_sale, free_item, cancelled, and
# stock_writein purely because of whatever sign their individual rows happen to carry, not
# because they represent genuine sales activity (see §7.8 in the report). Dropping them here,
# before any of the quality/category analysis below, keeps every downstream count and total
# based on real transactions only.
test_code_mask = df["StockCode"].isin(["TEST001", "TEST002"])
print(f"Test-product rows removed: {test_code_mask.sum():,}")

df = df.loc[~test_code_mask].reset_index(drop=True)
print(f"Rows remaining: {len(df):,}")

In [8]:
def column_extremes(column):
    values = df[column].dropna()
    if values.empty:
        return None, None
    if pd.api.types.is_object_dtype(values) or pd.api.types.is_string_dtype(values):
        values = values.astype("string")
    return values.min(), values.max()

extremes = [column_extremes(column) for column in df.columns]
extreme_values = pd.DataFrame(
    {
        "column": df.columns,
        "minimum": [minimum for minimum, _ in extremes],
        "maximum": [maximum for _, maximum in extremes],
        "non_null_values": [df[column].notna().sum() for column in df.columns],
    }
)

extreme_values

,column,minimum,maximum,non_null_values
0,Invoice,489434,C581569,1044848
1,StockCode,10002,m,1044848
2,Description,DOORMAT UNION JACK GUNS AND ROSES,wrongly sold sets,1040573
3,Quantity,-80995,80995,1044848
4,InvoiceDate,2009-12-01 07:45:00,2011-12-09 12:50:00,1044848
5,Price,-53594.36,38970.0,1044848
6,Customer ID,12346.0,18287.0,809561
7,Country,Australia,West Indies,1044848
8,source_sheet,Year 2009-2010,Year 2010-2011,1044848
9,Revenue,-168469.6,168469.6,1044848


In [ ]:
# The Quantity/Revenue extremes above are exactly +-80,995 -- check whether that's one order
# placed and then fully cancelled (as claimed in the report), or two unrelated coincidences.
extreme_rows = df.loc[df["Quantity"].abs() == 80995]
print(extreme_rows[["Invoice", "StockCode", "Description", "Quantity", "Price", "Revenue", "InvoiceDate"]].to_string(index=False))
print()
print("Combined net revenue from these two rows:", extreme_rows["Revenue"].sum())

## 7. Analyze Meaningless Numeric Values

In [9]:
meaningless_masks = {
    "negative_quantity": df["Quantity"] < 0,
    "zero_quantity": df["Quantity"] == 0,
    "negative_price": df["Price"] < 0,
    "zero_price": df["Price"] == 0,
    "negative_revenue": df["Revenue"] < 0,
    "zero_revenue": df["Revenue"] == 0,
}

meaningless_value_summary = pd.DataFrame(
    [
        {
            "issue": issue,
            "column": issue.rsplit("_", 1)[1].title(),
            "count": mask.sum(),
            "percentage_of_non_null": round(mask.sum() / df[column].notna().sum() * 100, 2),
        }
        for issue, mask in meaningless_masks.items()
        for column in [issue.rsplit("_", 1)[1].title()]
    ]
)

meaningless_flag_mask = pd.DataFrame(meaningless_masks).any(axis=1)
meaningless_rows = df.loc[meaningless_flag_mask].copy()
meaningless_rows["meaningless_flags"] = (
    pd.DataFrame(meaningless_masks, index=df.index)
    .loc[meaningless_flag_mask]
    .apply(lambda row: ", ".join(row.index[row]), axis=1)
)

display(meaningless_value_summary)
display(meaningless_rows.head(20))

,issue,column,count,percentage_of_non_null
0,negative_quantity,Quantity,22557,2.16
1,zero_quantity,Quantity,0,0.00
2,negative_price,Price,5,0.00
3,zero_price,Price,6024,0.58
4,negative_revenue,Revenue,19169,1.83
5,zero_revenue,Revenue,6024,0.58


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet,Revenue,meaningless_flags
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,Year 2009-2010,-35.40,"negative_quantity, negative_revenue"
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia,Year 2009-2010,-9.90,"negative_quantity, negative_revenue"
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia,Year 2009-2010,-17.00,"negative_quantity, negative_revenue"
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia,Year 2009-2010,-12.60,"negative_quantity, negative_revenue"
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,Year 2009-2010,-35.40,"negative_quantity, negative_revenue"
183,C489449,21871,SAVE THE PLANET MUG,-12,2009-12-01 10:33:00,1.25,16321.0,Australia,Year 2009-2010,-15.00,"negative_quantity, negative_revenue"
184,C489449,84946,ANTIQUE SILVER TEA GLASS ETCHED,-12,2009-12-01 10:33:00,1.25,16321.0,Australia,Year 2009-2010,-15.00,"negative_quantity, negative_revenue"
185,C489449,84970S,HANGING HEART ZINC T-LIGHT HOLDER,-24,2009-12-01 10:33:00,0.85,16321.0,Australia,Year 2009-2010,-20.40,"negative_quantity, negative_revenue"
186,C489449,22090,PAPER BUNTING RETRO SPOTS,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,Year 2009-2010,-35.40,"negative_quantity, negative_revenue"
196,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592.0,United Kingdom,Year 2009-2010,-12.75,"negative_quantity, negative_revenue"


## 8. Identify Invalid Rows

In [10]:
df["is_cancelled"] = df["Invoice"].str.upper().str.startswith("C", na=False)

missing_invoice = df["Invoice"].isna()
missing_stock_code = df["StockCode"].isna()
missing_price = df["Price"].isna()
non_numeric_quantity = df["Quantity"].isna()
invalid_date = df["InvoiceDate"].isna()

invalid_mask = (
    missing_invoice
    | missing_stock_code
    | missing_price
    | non_numeric_quantity
    | invalid_date
)
invalid_rows = df.loc[invalid_mask].copy()
invalid_rows["quality_flag"] = "invalid_core_data"
invalid_rows.loc[df.loc[invalid_mask, "Customer ID"].isna(), "quality_flag"] = "missing_customer_id"

print(f"Flagged rows: {len(invalid_rows):,}")
invalid_rows.head()

Flagged rows: 0


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet,Revenue,is_cancelled,quality_flag


In [ ]:
# Do cancellation invoices reference an existing original invoice? If stripping the "C" from a
# cancellation invoice number matched a real prior invoice, cancellations could be linked back
# to the specific order each one reverses.
cancel_invoices = df.loc[df["is_cancelled"], "Invoice"].unique()
stripped = pd.Series(cancel_invoices).str.upper().str[1:]
matches = stripped.isin(set(df["Invoice"])).sum()
print(f"Distinct cancellation invoices: {len(cancel_invoices):,}")
print(f"Stripped-prefix matches found among all Invoice values: {matches}")
print()

# Cancellations should always be negative-quantity by definition (that's the whole point of a
# reversal) -- check for exceptions.
positive_qty_cancellations = df.loc[df["is_cancelled"] & (df["Quantity"] > 0)]
print(f"Cancelled rows with POSITIVE quantity (should be none): {len(positive_qty_cancellations)}")
print(positive_qty_cancellations[["Invoice", "StockCode", "Description", "Quantity", "Price"]].to_string(index=False))

## 8b. Analyze Negative-Quantity Rows That Are Not Cancellations (Stock Adjustments)

In [ ]:
negative_noncancel_mask = (df["Quantity"] < 0) & ~df["is_cancelled"]
negative_noncancel = df.loc[negative_noncancel_mask].copy()

print(f"Negative-quantity, non-cancellation rows: {len(negative_noncancel):,}")
print()
print("Price value counts (all rows):")
print(negative_noncancel["Price"].value_counts(dropna=False))
print()
print("Missing Customer ID:", negative_noncancel["Customer ID"].isna().sum(), "of", len(negative_noncancel))
print()
print("Rows per invoice (do these ever share an invoice with a normal sale row?):")
mixed_invoices = set(negative_noncancel["Invoice"]) & set(df.loc[df["Quantity"] > 0, "Invoice"])
print("Invoices also containing a positive-quantity row:", len(mixed_invoices), "of", negative_noncancel["Invoice"].nunique())
print()
print("Top Description values (operational notes, not product names):")
print(negative_noncancel["Description"].value_counts(dropna=False).head(15))

# These rows carry Price == 0 and no Customer ID in every case, are never mixed into a
# normal order invoice, and their free-text Descriptions ("damages", "lost", "missing",
# "thrown away", "check", etc.) read as warehouse stock-control notes rather than
# customer activity. Flag them as stock adjustments, distinct from customer cancellations.
df["is_stock_adjustment"] = negative_noncancel_mask

print()
print("Flagged as is_stock_adjustment:", int(df["is_stock_adjustment"].sum()))
negative_noncancel.head(20)

# A small subset use non-standard StockCodes (not the normal 5-digit-plus-suffix product
# pattern) instead -- flagged here as a follow-up, not yet resolved (see §7.2 "Next step").
is_standard_code = negative_noncancel["StockCode"].str.match(r"^\d{5}[A-Za-z]{0,4}$", na=False)
print()
print(f"Stock-adjustment rows using a non-standard StockCode: {(~is_standard_code).sum()} of {len(negative_noncancel):,}")
print(negative_noncancel.loc[~is_standard_code, "StockCode"].value_counts().to_string())

## 8c. Analyze Zero-Price, Positive-Quantity Rows (Stock Write-ins vs. Free Items)

In [ ]:
zero_price_pos_qty_mask = (df["Price"] == 0) & (df["Quantity"] > 0) & ~df["is_cancelled"] & ~df["is_stock_adjustment"]
zero_price_pos_qty = df.loc[zero_price_pos_qty_mask].copy()

print(f"Zero-price, positive-quantity, non-cancelled, non-stock-adjustment rows: {len(zero_price_pos_qty):,}")
print()

has_customer = zero_price_pos_qty["Customer ID"].notna()
print("Split by Customer ID presence:")
print(has_customer.value_counts())
print()

print("--- Group 1: no Customer ID (stock write-in candidates) ---")
group1 = zero_price_pos_qty.loc[~has_customer]
print(f"Rows: {len(group1):,}")
print("Description value counts:")
print(group1["Description"].value_counts(dropna=False).head(15))
print()

# Do write-in rows ever land on the same invoice as a normally priced row? If so, excluding the
# write-in line can't be double-counted or lose revenue, since the invoice's real total lives on
# its other, priced rows.
mixed_writein_invoices = set(group1["Invoice"]) & set(df.loc[df["Price"] > 0, "Invoice"])
print(f"Write-in rows sharing an invoice with a normally priced row: {len(mixed_writein_invoices):,} of {group1['Invoice'].nunique():,} write-in invoices")
print()

print("--- Group 2: has Customer ID (free item candidates) ---")
group2 = zero_price_pos_qty.loc[has_customer]
print(f"Rows: {len(group2):,}")
print("Description value counts:")
print(group2["Description"].value_counts(dropna=False).head(15))
print()
print(group2[["Invoice", "StockCode", "Description", "Quantity", "Customer ID"]].head(15))

# Group 1 mirrors is_stock_adjustment (which covers negative-quantity write-offs): no customer,
# no price, generic/blank operational notes -- stock being corrected back into inventory rather
# than sold. Group 2 has a real Customer ID attached to a real product -- a genuine transaction,
# just given away for free -- so it's kept in sales/orders rather than excluded.
df["is_stock_writein"] = zero_price_pos_qty_mask & df["Customer ID"].isna()
df["is_free_item"] = zero_price_pos_qty_mask & df["Customer ID"].notna()

print()
print("Flagged as is_stock_writein:", int(df["is_stock_writein"].sum()))
print("Flagged as is_free_item:", int(df["is_free_item"].sum()))

## 8d. Analyze "Adjust Bad Debt" Rows (Invoice Prefix "A")

In [ ]:
# Neither is_cancelled ("C" prefix) nor is_stock_adjustment (negative quantity) catches these --
# invoices prefixed "A" are financial bad-debt write-offs (StockCode "B"), not a stock movement
# or a customer cancellation, and were previously sitting silently inside sales_rows/gross_revenue.
df["is_bad_debt_adjustment"] = df["Invoice"].str.upper().str.startswith("A", na=False)
bad_debt_rows = df.loc[df["is_bad_debt_adjustment"]].copy()

print(f"Bad debt adjustment rows: {len(bad_debt_rows)}")
print(bad_debt_rows[["Invoice", "StockCode", "Description", "Quantity", "Price", "Customer ID", "InvoiceDate"]].to_string(index=False))
print()
print("Net revenue impact:", round(bad_debt_rows["Revenue"].sum(), 2))
print()
print("Months containing a bad debt write-off:", sorted(bad_debt_rows["InvoiceDate"].dt.to_period("M").astype(str).unique()))

## 8e. Full Data Taxonomy (Cross-Examination of All Record Categories)

Each row is assigned to exactly one category by checking the rules below **in order** — a row gets the first category whose rule it matches (implemented as the if/elif chain in `categorize()` below). Anything matching none of the first five rules falls through to `normal_sale`.

| Priority | Category | Rule | Counted as a sale? |
|---|---|---|---|
| 1 | `cancelled` | `Invoice` starts with "C" | No — excluded from `sales_rows`, but netted back in for net revenue (§10) |
| 2 | `bad_debt_adjustment` | `Invoice` starts with "A" | No — excluded entirely |
| 3 | `stock_adjustment` | `Quantity < 0` (and not already `cancelled`/`bad_debt_adjustment`) | No — excluded entirely |
| 4 | `stock_writein` | `Price = 0` and `Quantity > 0` and `Customer ID` missing | No — excluded entirely |
| 5 | `free_item` | `Price = 0` and `Quantity > 0` and `Customer ID` present | **Yes** — kept in, real order at £0 |
| 6 *(fallback)* | `normal_sale` | Everything left over (in practice, `Quantity > 0` and `Price > 0`) | **Yes** |

In [14]:
def sign(series):
    return pd.cut(
        series,
        bins=[-float("inf"), -1e-9, 1e-9, float("inf")],
        labels=["negative", "zero", "positive"],
    )


def categorize(row):
    if row["is_cancelled"]:
        return "cancelled"
    if row["is_bad_debt_adjustment"]:
        return "bad_debt_adjustment"
    if row["is_stock_adjustment"]:
        return "stock_adjustment"
    if row["is_stock_writein"]:
        return "stock_writein"
    if row["is_free_item"]:
        return "free_item"
    return "normal_sale"


df["qty_sign"] = sign(df["Quantity"])
df["price_sign"] = sign(df["Price"])
df["category"] = df.apply(categorize, axis=1)

data_taxonomy = (
    df.groupby(["category", "qty_sign", "price_sign"], observed=True)
    .size()
    .reset_index(name="count")
    .sort_values(["category", "count"], ascending=[True, False])
)

print(f"Total rows across all categories: {data_taxonomy['count'].sum():,} (dataset has {len(df):,})")
data_taxonomy

Total rows across all categories: 1,044,848 (dataset has 1,044,848)


,category,qty_sign,price_sign,count
0,bad_debt_adjustment,positive,negative,5
1,bad_debt_adjustment,positive,positive,1
2,cancelled,negative,positive,19164
3,cancelled,positive,positive,1
4,free_item,positive,zero,70
5,normal_sale,positive,positive,1019653
6,stock_adjustment,negative,zero,3393
7,stock_writein,positive,zero,2561


## 8f. Duplicate Line-Item Flag (`is_duplicate_line`)

In [ ]:
# Full-row duplicates: same Invoice, StockCode, Description, Quantity, InvoiceDate, Price,
# Customer ID, and Country. The sheet-overlap duplication (§3b) is already removed, so what's
# left here is duplication WITHIN a single sheet -- e.g. the same product logged as two separate
# single-unit lines rather than one line at a higher quantity. This flag doesn't drop anything;
# it's for visibility, since it isn't clear this is an error rather than how orders were entered.
dedup_cols = ["Invoice", "StockCode", "Description", "Quantity", "InvoiceDate", "Price", "Customer ID", "Country"]
df["is_duplicate_line"] = df.duplicated(subset=dedup_cols, keep=False)

duplicate_lines = df.loc[df["is_duplicate_line"]].copy()
print(f"Duplicate-flagged rows: {len(duplicate_lines):,}")
print(f"Distinct duplicate-content groups: {duplicate_lines.drop_duplicates(subset=dedup_cols).shape[0]:,}")
print(f"Distinct invoices affected: {duplicate_lines['Invoice'].nunique():,}")
print()

group_sizes = duplicate_lines.groupby(dedup_cols, observed=True).size()
print("How many times each duplicate group repeats:")
print(group_sizes.value_counts().sort_index())
print()

extra_copies = duplicate_lines.copy()
extra_copies["copy_number"] = extra_copies.groupby(dedup_cols, observed=True).cumcount()
extra_only = extra_copies.loc[extra_copies["copy_number"] > 0]
print(f"Revenue represented by the 'extra' (2nd+) copies in each group: £{extra_only['Revenue'].sum():,.2f}")
print(f"Duplicate rows that are also cancellations: {duplicate_lines['is_cancelled'].sum():,} of {len(duplicate_lines):,}")
print()

print("Quantity distribution among duplicated lines:")
print(duplicate_lines["Quantity"].describe())
print(f"Duplicate lines with Quantity == 1: {(duplicate_lines['Quantity'] == 1).sum():,} of {len(duplicate_lines):,}")
print()
print("Customers with the most duplicate-flagged rows (likely wholesale/bulk buyers logging repeat single-unit lines):")
print(duplicate_lines["Customer ID"].value_counts(dropna=False).head(10))

# Is "wholesale/bulk buyer" a fair read, or just noise from a handful of customers with lots of
# orders generally? Rank every customer by total revenue and total order count, then check where
# the top duplicate-line customers fall in that ranking -- if they're wholesale buyers, they
# should sit at the extreme high end on both measures, not just have a lot of duplicate rows.
customer_profile = df.groupby("Customer ID").agg(
    total_rows=("Revenue", "size"),
    total_revenue=("Revenue", "sum"),
    total_orders=("Invoice", "nunique"),
).reset_index()
customer_profile["revenue_percentile"] = customer_profile["total_revenue"].rank(pct=True)
customer_profile["orders_percentile"] = customer_profile["total_orders"].rank(pct=True)

top_duplicate_customers = duplicate_lines["Customer ID"].value_counts().head(10).index.dropna()
wholesale_check = (
    customer_profile.loc[customer_profile["Customer ID"].isin(top_duplicate_customers)]
    .merge(duplicate_lines["Customer ID"].value_counts().rename("duplicate_rows"), left_on="Customer ID", right_index=True)
    .sort_values("total_revenue", ascending=False)
)
print()
print("Where the top duplicate-line customers rank among ALL customers by revenue and order count:")
print(wholesale_check.to_string(index=False))

# Are duplicates confined to a narrow window (pointing at a system bug active only then), or
# spread across the whole dataset (more consistent with an ongoing order-entry practice)?
df["month"] = df["InvoiceDate"].dt.to_period("M").astype("string")
duplicate_lines["month"] = duplicate_lines["InvoiceDate"].dt.to_period("M").astype("string")

monthly_duplicates = pd.concat(
    [df.groupby("month").size().rename("total_rows"), duplicate_lines.groupby("month").size().rename("duplicate_rows")],
    axis=1,
).fillna(0)
monthly_duplicates["duplicate_rate_pct"] = (monthly_duplicates["duplicate_rows"] / monthly_duplicates["total_rows"] * 100).round(2)

print()
print("Duplicate rows by month:")
print(monthly_duplicates.to_string())
print()
print("Duplicate row date range:", duplicate_lines["InvoiceDate"].min(), "->", duplicate_lines["InvoiceDate"].max())
print("Full dataset date range: ", df["InvoiceDate"].min(), "->", df["InvoiceDate"].max())
print("Months with zero duplicate rows:", (monthly_duplicates["duplicate_rows"] == 0).sum(), "of", len(monthly_duplicates))
# Same question at day-level granularity: is there one specific day dominating (a one-off
# system incident) or is it spread across nearly every day the business operated?
date_counts = duplicate_lines["InvoiceDate"].dt.date.value_counts()
print()
print(f"Dates with at least one duplicate row: {len(date_counts)} of {df['InvoiceDate'].dt.date.nunique()} total dates in the dataset")
print()
print("Top 20 dates by duplicate row count:")
print(date_counts.head(20))
print()
print("Distribution of duplicate-row-count per date:")
print(date_counts.describe())

## 8g. Non-Product Stock Code Flag (`is_non_product_code`)

In [16]:
# Standard product codes are 5 digits with an optional short letter suffix (e.g. "85123A").
# Anything else is a bookkeeping/operational code riding along in the same StockCode column --
# postage, manual adjustments, discounts, fees, gift vouchers, test rows, etc.
df["is_non_product_code"] = ~df["StockCode"].str.match(r"^\d{5}[A-Za-z]{0,4}$", na=False)

# category (from Section 8e) already encodes which flag -- if any -- takes a row out of
# sales_rows: "normal_sale" and "free_item" are counted as a sale; every other category is
# excluded.
non_product = df.loc[df["is_non_product_code"]].copy()
non_product["counted_as_sale"] = non_product["category"].isin(["normal_sale", "free_item"])

print(f"Non-product-code rows: {len(non_product):,} across {non_product['StockCode'].nunique()} distinct codes")
print()

code_breakdown = (
    non_product.groupby(["StockCode", "category"], observed=True)
    .agg(rows=("StockCode", "size"), revenue=("Revenue", "sum"))
    .reset_index()
    .sort_values(["StockCode", "rows"], ascending=[True, False])
)
print(code_breakdown.to_string(index=False))
print()

print("Rolled up by StockCode -- total rows, net revenue, and how much of that revenue currently")
print("counts as a sale vs. is excluded by an existing flag:")
rollup = non_product.groupby("StockCode", observed=True).apply(
    lambda g: pd.Series(
        {
            "rows": len(g),
            "net_revenue": g["Revenue"].sum(),
            "revenue_counted_as_sale": g.loc[g["counted_as_sale"], "Revenue"].sum(),
            "revenue_excluded": g.loc[~g["counted_as_sale"], "Revenue"].sum(),
        }
    ),
    include_groups=False,
).sort_values("rows", ascending=False)
print(rollup.to_string())

Non-product-code rows: 5,993 across 63 distinct codes

   StockCode            category  rows     revenue
     47503J          normal_sale     1      16.130
      ADJUST         normal_sale    36    8897.930
      ADJUST           cancelled    31   -2062.690
     ADJUST2         normal_sale     3     731.050
   AMAZONFEE           cancelled    33 -241988.300
   AMAZONFEE         normal_sale     3   20467.800
           B bad_debt_adjustment     6 -147614.080
BANK CHARGES           cancelled    66  -36001.490
BANK CHARGES         normal_sale    34     519.241
          C2         normal_sale   267   13426.000
          C2           cancelled     7    -290.000
          C2       stock_writein     3       0.000
          C3    stock_adjustment     1       0.000
        CRUK           cancelled    16   -7933.430
           D           cancelled   168  -13277.520
           D         normal_sale     5     397.890
    DCGS0003         normal_sale    13      32.590
    DCGS0003    stock_adjus

In [ ]:
# Group the near-duplicate codes together (M/m are the same thing typed differently, likewise
# ADJUST/ADJUST2). TEST001/TEST002 no longer appear here -- they were dropped from df entirely
# in §6b as literal QA rows, not real transactions.
code_groups = {
    "M": "M / m", "m": "M / m",
    "ADJUST": "ADJUST / ADJUST2", "ADJUST2": "ADJUST / ADJUST2",
}
non_product["code_group"] = non_product["StockCode"].map(lambda c: code_groups.get(c, c))

named_groups = [
    "POST", "DOT", "M / m", "C2", "D", "BANK CHARGES", "S",
    "ADJUST / ADJUST2", "AMAZONFEE", "CRUK", "B",
]
named = non_product.loc[non_product["code_group"].isin(named_groups)]
rest = non_product.loc[~non_product["code_group"].isin(named_groups)]

category_pivot = (
    named.groupby(["code_group", "category"], observed=True)
    .agg(rows=("Revenue", "size"), revenue=("Revenue", "sum"))
    .round(2)
)
print("Rows / revenue per named code, split by category (matches the §7.8 table in the report):")
for group in named_groups:
    print(f"--- {group} ---")
    print(category_pivot.loc[group].to_string())
    print()

print(f"--- remaining {rest['StockCode'].nunique()} codes, combined ---")
print(rest.groupby("category", observed=True).agg(rows=("Revenue", "size"), revenue=("Revenue", "sum")).round(2).to_string())

# The most common raw Description text per named code group -- backs the "Raw Description text"
# column of the §7.8 table (e.g. POST -> "POSTAGE", M / m -> "Manual").
print()
print("Most common raw Description text per named code group:")
print(named.groupby("code_group", observed=True)["Description"].agg(lambda s: s.value_counts().index[0]).to_string())

## 8h. Multiple Descriptions per Product Code (`has_variant_description`)

In [ ]:
# Restricted to genuine product codes (is_non_product_code == False) -- non-product codes like
# "ADJUST"/"M" are *expected* to carry many different free-text notes by design (§7.8), so
# including them here would just be noise, not a data-quality finding about products.
product_rows = df.loc[~df["is_non_product_code"]].dropna(subset=["Description"])

n_distinct = product_rows.groupby("StockCode")["Description"].nunique()
multi_desc_codes = n_distinct.loc[n_distinct > 1].index

print(f"Product codes with more than one distinct Description: {len(multi_desc_codes):,} of {n_distinct.shape[0]:,}")

def modal_share(descriptions):
    counts = descriptions.value_counts()
    return counts.iloc[0] / counts.sum()

shares = (
    product_rows.loc[product_rows["StockCode"].isin(multi_desc_codes)]
    .groupby("StockCode")["Description"]
    .apply(modal_share)
)

# A code where one wording covers >=95% of its rows is one real product name plus a handful of
# stray operational notes ("missing", "found", "wrongly coded-23343", ...) riding in the
# Description field -- not a naming problem. Below that threshold the wordings are genuinely
# competing, which usually means the product was renamed/reworded partway through the two-year
# window (e.g. "PINK POLKADOT PLATE" vs "PINK SPOTTY PLATE" for the same StockCode).
stray_notes = shares.loc[shares >= 0.95]
renamed_products = shares.loc[shares < 0.95].sort_values()

df["has_variant_description"] = df["StockCode"].isin(renamed_products.index)

print(f"  - one dominant name + stray one-off notes: {len(stray_notes):,} codes")
print(f"  - meaningfully split between multiple wordings (likely renamed/reworded product): {len(renamed_products):,} codes")
print()

stray_note_sizes = (
    product_rows.loc[product_rows["StockCode"].isin(stray_notes.index)]
    .groupby("StockCode").size().sort_values(ascending=False)
)
stray_examples = (
    product_rows.loc[product_rows["StockCode"].isin(stray_note_sizes.index[:15])]
    .groupby("StockCode")["Description"]
    .apply(lambda s: s.value_counts().to_dict())
    .loc[stray_note_sizes.index[:15]]
    .rename("descriptions")
    .reset_index()
)
print("Largest stray-note codes (one dominant name + a few one-off notes) -- top 15 by row count:")
display(stray_examples.head(15))

description_variants = (
    product_rows.loc[product_rows["StockCode"].isin(renamed_products.index)]
    .groupby("StockCode")["Description"]
    .apply(lambda s: s.value_counts().to_dict())
    .loc[renamed_products.index]
    .rename("descriptions")
    .reset_index()
)
print("Most evenly split codes (same product, different wording over time) -- top 15:")
display(description_variants.head(15))

# Resolve each ambiguous code to a single canonical Description, using a different rule per
# group since they represent different situations:
#   - stray-note codes: the dominant wording IS the product name, so use the most frequent
#     (modal) Description and discard the one-off notes.
#   - renamed/reworded codes: there's no single "correct" wording, so use whichever Description
#     was actually in use most recently (latest InvoiceDate), since that reflects current
#     catalog naming rather than an arbitrary pick.
modal_description = (
    product_rows.loc[product_rows["StockCode"].isin(stray_notes.index)]
    .groupby("StockCode")["Description"]
    .agg(lambda s: s.value_counts().idxmax())
)
latest_description = (
    product_rows.loc[product_rows["StockCode"].isin(renamed_products.index)]
    .sort_values("InvoiceDate")
    .groupby("StockCode")["Description"]
    .last()
)
canonical_description = pd.concat([modal_description, latest_description])

original_description = df["Description"]
mapped_description = df["StockCode"].map(canonical_description)
rows_relabelled = (mapped_description.notna() & (mapped_description != original_description)).sum()
df["Description"] = mapped_description.fillna(original_description)

print(
    f"Rows relabelled with a canonical Description: {rows_relabelled:,} across "
    f"{len(canonical_description):,} StockCodes ({len(modal_description):,} resolved to the "
    f"most-frequent wording, {len(latest_description):,} resolved to the latest wording)"
)

## 8i. Non-Country Flag (`is_non_country`)

In [ ]:
# A handful of "Country" values aren't actual countries: "Unspecified" is a genuine unknown,
# while "European Community", "Channel Islands", and "West Indies" are a political bloc, a
# dependency, and a multi-country region respectively -- none map cleanly onto a single nation.
# ("EIRE" and "RSA" ARE real countries -- just Irish/Afrikaans-derived names for Ireland and
# South Africa -- so they're left alone.)
non_country_values = ["Unspecified", "European Community", "Channel Islands", "West Indies"]
df["is_non_country"] = df["Country"].isin(non_country_values)

print(f"Non-country rows: {df['is_non_country'].sum():,} ({df['is_non_country'].mean() * 100:.2f}% of rows)")
display(
    df.loc[df["is_non_country"]]
    .groupby("Country", observed=True)
    .agg(rows=("Country", "size"), revenue=("Revenue", "sum"))
    .sort_values("rows", ascending=False)
)

# Flagged for visibility only -- the revenue is real, so these rows stay in Top Countries (§6)
# under their own label rather than being dropped or forced into a single-country bucket.

## 9. Review the First Rows and Missing Values

In [18]:
display(df.head(10))
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_values"))
display(df["Country"].value_counts(dropna=False).head(20).to_frame("row_count"))

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet,Revenue,is_cancelled,is_stock_adjustment,is_stock_writein,is_free_item,is_bad_debt_adjustment,qty_sign,price_sign,category,is_duplicate_line,month,is_non_product_code
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Year 2009-2010,83.4,False,False,False,False,False,positive,positive,normal_sale,False,2009-12,False
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010,81.0,False,False,False,False,False,positive,positive,normal_sale,False,2009-12,False
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010,81.0,False,False,False,False,False,positive,positive,normal_sale,False,2009-12,False
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Year 2009-2010,100.8,False,False,False,False,False,positive,positive,normal_sale,False,2009-12,False
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010,30.0,False,False,False,False,False,positive,positive,normal_sale,False,2009-12,False
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom,Year 2009-2010,39.6,False,False,False,False,False,positive,positive,normal_sale,False,2009-12,False
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010,30.0,False,False,False,False,False,positive,positive,normal_sale,False,2009-12,False
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom,Year 2009-2010,59.5,False,False,False,False,False,positive,positive,normal_sale,False,2009-12,False
8,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom,Year 2009-2010,30.6,False,False,False,False,False,positive,positive,normal_sale,False,2009-12,False
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,13085.0,United Kingdom,Year 2009-2010,45.0,False,False,False,False,False,positive,positive,normal_sale,False,2009-12,False


,missing_values
Customer ID,235287
Description,4275
Invoice,0
Quantity,0
StockCode,0
InvoiceDate,0
Price,0
Country,0
source_sheet,0
Revenue,0


,row_count
Country,
United Kingdom,959983
EIRE,17689
Germany,17363
France,14059
Netherlands,5138
Spain,3766
Switzerland,3183
Belgium,3111
Portugal,2540


## 10. Aggregate Sales Metrics

In [ ]:
non_transactional = df["is_stock_adjustment"] | df["is_stock_writein"] | df["is_bad_debt_adjustment"]

# is_free_item is NOT excluded here -- it's a real order for a real customer, just at £0.
sales_rows = df.loc[~df["is_cancelled"] & ~non_transactional & df["Revenue"].notna()].copy()
sales_rows["month"] = sales_rows["InvoiceDate"].dt.to_period("M").astype("string")

# Cancellations are not reliably linkable to a specific original order (see §7.1 decision),
# but the money they return is real, so revenue must be NET of cancellations rather than
# computed on sales rows alone -- otherwise cancelled purchases are counted as revenue that
# was never actually kept. revenue_rows = sales + cancellations, non-transactional rows
# (stock adjustments/write-ins/bad debt) excluded since they aren't customer transactions.
revenue_rows = df.loc[~non_transactional & df["Revenue"].notna()].copy()
revenue_rows["month"] = revenue_rows["InvoiceDate"].dt.to_period("M").astype("string")

gross_revenue = sales_rows["Revenue"].sum()
cancellation_revenue = df.loc[df["is_cancelled"], "Revenue"].sum()
bad_debt_revenue = df.loc[df["is_bad_debt_adjustment"], "Revenue"].sum()
net_revenue = revenue_rows["Revenue"].sum()

monthly_revenue = revenue_rows.groupby("month", as_index=False)["Revenue"].sum().sort_values("month")
top_products = (
    revenue_rows.groupby("Description", dropna=False)["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(20)
    .rename("Revenue")
    .reset_index()
)
top_countries = (
    revenue_rows.groupby("Country", dropna=False)["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(20)
    .rename("Revenue")
    .reset_index()
)
top_countries["share_of_net_revenue_pct"] = (top_countries["Revenue"] / net_revenue * 100).round(1)

metrics = pd.Series(
    {
        "gross_revenue": gross_revenue,
        "cancellation_revenue": cancellation_revenue,
        "bad_debt_revenue": bad_debt_revenue,
        "net_revenue": net_revenue,
        "orders": sales_rows["Invoice"].nunique(),
        "customers": revenue_rows["Customer ID"].nunique(),
        "cancelled_rows": int(df["is_cancelled"].sum()),
        "stock_adjustment_rows": int(df["is_stock_adjustment"].sum()),
        "stock_writein_rows": int(df["is_stock_writein"].sum()),
        "free_item_rows": int(df["is_free_item"].sum()),
        "bad_debt_rows": int(df["is_bad_debt_adjustment"].sum()),
        "duplicate_flagged_rows": int(df["is_duplicate_line"].sum()),
        "non_product_code_rows": int(df["is_non_product_code"].sum()),
        "flagged_rows": len(invalid_rows),
    },
    name="value",
)
display(metrics.to_frame())
print("Net revenue by month (full 25 months -- backs the §4 seasonality claims):")
display(monthly_revenue)
display(top_products)

# How much of the top-20 products total is shipping (POST/DOT), not merchandise?
postage_revenue = top_products.loc[top_products["Description"].isin(["POSTAGE", "DOTCOM POSTAGE"]), "Revenue"].sum()
print(f"POSTAGE + DOTCOM POSTAGE combined: £{postage_revenue:,.0f} ({postage_revenue / top_products['Revenue'].sum() * 100:.1f}% of the top-20 total)")

display(top_countries)

## 11. Save Processed Outputs

In [ ]:
# Intentionally left empty for now -- the previous bulk CSV export here has been removed
# since those specific output datasets are being replaced. Add your own writes below.

In [21]:
# Scratch cell -- just for a quick visual look at the duplicate-flagged rows. Delete later.
duplicate_lines[duplicate_lines["Invoice"] == "492525"].sort_values(dedup_cols)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,source_sheet,Revenue,is_cancelled,is_stock_adjustment,is_stock_writein,is_free_item,is_bad_debt_adjustment,qty_sign,price_sign,category,is_duplicate_line,month
37919,492525,84255B,ORANGE COCKTAIL GLASS LAMP,1,2009-12-17 12:32:00,2.10,12748.0,United Kingdom,Year 2009-2010,2.10,False,False,False,False,False,positive,positive,normal_sale,True,2009-12
37921,492525,84255B,ORANGE COCKTAIL GLASS LAMP,1,2009-12-17 12:32:00,2.10,12748.0,United Kingdom,Year 2009-2010,2.10,False,False,False,False,False,positive,positive,normal_sale,True,2009-12
37934,492525,84990,60 GOLD AND SILVER FAIRY CAKE CASES,1,2009-12-17 12:32:00,0.55,12748.0,United Kingdom,Year 2009-2010,0.55,False,False,False,False,False,positive,positive,normal_sale,True,2009-12
37938,492525,84990,60 GOLD AND SILVER FAIRY CAKE CASES,1,2009-12-17 12:32:00,0.55,12748.0,United Kingdom,Year 2009-2010,0.55,False,False,False,False,False,positive,positive,normal_sale,True,2009-12


In [22]:
# Scratch cell -- worked example for invoice 492525 (customer 12748.0), showing how
# "ORANGE COCKTAIL GLASS LAMP" (StockCode 84255B) was entered as two separate Quantity=1
# lines with a different product in between, rather than one line at Quantity=2. Delete later.
example_invoice = df.loc[df["Invoice"] == "492525"]
example_invoice[["Invoice", "StockCode", "Description", "Quantity", "Price", "InvoiceDate", "Customer ID", "is_duplicate_line"]]

,Invoice,StockCode,Description,Quantity,Price,InvoiceDate,Customer ID,is_duplicate_line
37918,492525,79323W,WHITE CHERRY LIGHTS,1,6.75,2009-12-17 12:32:00,12748.0,False
37919,492525,84255B,ORANGE COCKTAIL GLASS LAMP,1,2.10,2009-12-17 12:32:00,12748.0,True
37920,492525,21591,COSY HOUR CIGAR BOX MATCHES,1,1.25,2009-12-17 12:32:00,12748.0,False
37921,492525,84255B,ORANGE COCKTAIL GLASS LAMP,1,2.10,2009-12-17 12:32:00,12748.0,True
37922,492525,84986B,SET OF 36 SILVER PAPER DOILIES,1,1.45,2009-12-17 12:32:00,12748.0,False
37923,492525,70007,HI TEC ALPINE HAND WARMER,1,1.65,2009-12-17 12:32:00,12748.0,False
37924,492525,21412,VINTAGE GOLD TINSEL REEL,2,0.42,2009-12-17 12:32:00,12748.0,False
37925,492525,37495,FAIRY CAKE BIRTHDAY CANDLE SET,1,3.75,2009-12-17 12:32:00,12748.0,False
37926,492525,85098B,BLUE FLYING SINGING CANARY,1,3.75,2009-12-17 12:32:00,12748.0,False
37927,492525,21821,GLITTER STAR GARLAND WITH BELLS,1,3.75,2009-12-17 12:32:00,12748.0,False
